# 🌊 AquaGuard AI — Day 23: Single-Pass Held-Out Test Evaluation
- **Date:** 11 Sep 2026
- **Phase:** Phase 4: Production Delivery & Verification
- **Objective:** Unblind the quarantined Test Lockbox partition for the first and only time; evaluate final production performance.
- **Inputs Consumed:** `artifacts/day22/production_bundle.joblib`, `data/raw/water_potability.csv`
- **Outputs Produced:** `artifacts/day23/test_performance.json`, `artifacts/day23/test_pr_and_calibration.png`
- **Deliverable Satisfied:** Unbiased test set metrics, PR-AUC / calibration curves, and latency profiling.

In [1]:
import os, sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").resolve().name == "Water Potability" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import time
import numpy as np
import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_artifact, save_artifact
from src.evaluation import compute_classification_metrics
from src.plotting import plot_pr_and_calibration
from src.data.preprocessing import DataPreprocessor
from src.features.engineer import FeatureEngineer

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day23")

# Load frozen production bundle
bundle = load_artifact("artifacts/day22/production_bundle.joblib", required_by_day="Day 22")
print(f"Loaded frozen production bundle: {bundle['model_name']} (Trained: {bundle['timestamp']})")

Loaded frozen production bundle: Random_Forest (Trained: 2026-10-19T12:21:50.329740+00:00)


## 1. Single-Pass Unblinding of Test Lockbox
Load Test Lockbox partition. Evaluate metrics once without retraining or tuning.

In [2]:
df_tabular = pd.read_csv(config["paths"]["raw_tabular_data"])
df_fe = FeatureEngineer.engineer_track_b_features(df_tabular)

prep = DataPreprocessor(random_state=config["seed"])
_, X_lock, _, y_lock = prep.split_track_b(df_fe, target_col="Potability", test_size=0.20)

pipeline = bundle["pipeline"]
clf = bundle["classifier"]

# Measure inference latency
t0 = time.perf_counter()
X_lock_trans = pipeline.transform(X_lock)
probs = clf.predict_proba(X_lock_trans)
preds = clf.predict(X_lock_trans)
t1 = time.perf_counter()

total_time_ms = (t1 - t0) * 1000.0
latency_per_sample_ms = total_time_ms / len(X_lock)

# Compute full metrics
test_metrics = compute_classification_metrics(y_lock.values, preds, probs)
test_metrics["inference_time_ms_per_sample"] = round(latency_per_sample_ms, 3)
test_metrics["lockbox_sample_count"] = len(y_lock)

save_artifact(test_metrics, "artifacts/day23/test_performance.json")
print("Single-Pass Lockbox Test Metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

2026-10-19 17:52:46 | INFO     | FeatureEngineer:52 - Engineered Track B features: Shape evolved from (3276, 10) to (3276, 19)
2026-10-19 17:52:47 | INFO     | DataPreprocessor:126 - Track B Split complete: Train=2620 (Positive=39.01%), Test Lockbox=656 (Positive=39.02%)


2026-10-19 17:52:48 | INFO    | IO - Saved artifact to artifacts\day23\test_performance.json


Single-Pass Lockbox Test Metrics:
  accuracy: 0.654
  precision: 0.6593
  recall: 0.2344
  f1_score: 0.3458
  f2_score: 0.2691
  brier_score: 0.2159
  roc_auc: 0.6659
  pr_auc: 0.5943
  inference_time_ms_per_sample: 2.364
  lockbox_sample_count: 656


## 2. Precision-Recall & Calibration Diagnostics
Verify probability reliability on unseen test samples.

In [3]:
plot_pr_and_calibration(
    y_lock.values,
    probs,
    model_name=bundle["model_name"],
    save_path="artifacts/day23/test_pr_and_calibration.png"
)
print("Diagnostic curves saved to artifacts/day23/test_pr_and_calibration.png")

2026-10-19 17:52:52 | INFO    | Plotting - Saved figure to artifacts\day23\test_pr_and_calibration.png


Diagnostic curves saved to artifacts/day23/test_pr_and_calibration.png


## 📌 Day 23 Execution Findings & Sign-Off
- **Key Finding:** Test PR-AUC = 0.5943, Brier = 0.2159. Unbiased results match established benchmarks.
- **Artifacts Saved:** `artifacts/day23/test_performance.json`, `artifacts/day23/test_pr_and_calibration.png`.
- **Handoff:** Ready for Day 25 error failure analysis (after Day 24 peer audit).